## 01 - Assemble 24-h recordings

This notebook assembles hourly Neuroplayer EDF files into 24-h recordings for each animal and signal.

Electrical signals and temperature are calibrated using `calibration.py`, while activity values are preserved. Missing intervals are recorded in JSON sidecars.

**How to run:** Set the paths and source-value settings in the configuration cell, then run the notebook. The optional verification section can be run separately.

In [ ]:
import os
import sys
import re
import time
import numpy as np
import pyedflib
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from concurrent.futures import ProcessPoolExecutor, as_completed

# ── ADJUST this path to wherever you saved edf_writer_worker.py ──
WORKER_DIR = Path("/path/to/folder/with/edf_writer_worker.py_and_calibration.py")  # <── CHANGE THIS LINE

if not WORKER_DIR.exists():
    raise FileNotFoundError(
        f"\n\n"
        f"  WORKER_DIR not found: {WORKER_DIR}\n"
        f"  Please update the WORKER_DIR variable above to the folder\n"
        f"  that contains 'edf_writer_worker.py'.\n"
    )

_worker_script = WORKER_DIR / "edf_writer_worker.py"
if not _worker_script.exists():
    raise FileNotFoundError(
        f"\n\n"
        f"  'edf_writer_worker.py' not found inside: {WORKER_DIR}\n"
        f"  Make sure the file exists there, or update WORKER_DIR above.\n"
    )

if str(WORKER_DIR) not in sys.path:
    sys.path.insert(0, str(WORKER_DIR))

from edf_writer_worker import write_single_channel_edf
import calibration
print(f"Worker loaded from: {_worker_script}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
RECEIVER_ID = "ALT158" # <── CHANGE receiver (ALT157, ALT158 or ALT160)
DATA_DIR = Path(f"/path/to/source/{RECEIVER_ID}")  # <── CHANGE THIS LINE
DATASET_ROOT = Path("/path/to/dataset")  # <── CHANGE THIS LINE
OUTPUT_DIR = DATASET_ROOT / "recordings-24h"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# rat1, rat2, rat3 of the receiver -> animal IDs of the dataset (recordings-24h/Rat01 ... Rat08)
RAT_IDS = {"ALT157": ["Rat01", "Rat02", "Rat03"],
           "ALT158": ["Rat04", "Rat05", "Rat06"],
           "ALT160": ["Rat07", "Rat08"]}[RECEIVER_ID]

EXPECTED_DURATION = 3600  # seconds per source file
WINDOW_DURATION = 86400   # 24 hours in seconds
KEEP_LAST_WINDOW_REAL_DURATION = True
WRITE_NAN_SIDECAR = True # EDF cannot store NaN values, so missing-data gaps are written as zeros; the JSON sidecar records where NaNs/gaps existed before EDF export.

# File selection mode:
#   "all"   -> process all discovered files
#   "range" -> process only files n..p (1-based, inclusive)
FILE_SELECTION_MODE = "all" # <── CHANGE THIS LINE
FILE_RANGE_N = 1
FILE_RANGE_P = 4

# E-file channel layout per rat: (name, expected sampling rate)
CHANNELS_PER_RAT_E = [
    ("EMG",         128),
    ("ECG",         256),
    ("EGG",          64),
    ("EEG",         512),
    ("Temperature",  64),
]
CHANNEL_NAMES_E = [c[0] for c in CHANNELS_PER_RAT_E]
N_CH_PER_RAT_E = len(CHANNELS_PER_RAT_E)  # 5
N_CH_PER_RAT_A = 5  # A-files also have 5 channels per rat (all identical)

# Content of the source files: "counts" (16-bit ADC counts, converted) or "calibrated" (already uV / degC, not converted)
SOURCE_VALUES = {"EMG": "counts", "ECG": "counts", "EGG": "counts", "EEG": "counts", "Temperature": "counts"}  # <── CHANGE THIS LINE
TEMPERATURE_TABLE_PATH = Path("/path/to/temp_calibration_final.xlsx")  # <── CHANGE THIS LINE
calibration.SOURCE_VALUES.update(SOURCE_VALUES)
TEMPERATURE_TABLE = calibration.load_temperature_table(TEMPERATURE_TABLE_PATH)  # None when temperature is already calibrated

# Local timezone for filenames
LOCAL_TZ = ZoneInfo("Europe/Paris")

# Number of parallel workers
N_WORKERS = max(1, os.cpu_count() - 2)
print(f"Using {N_WORKERS} parallel workers (CPU cores: {os.cpu_count()})")
print(f"File selection mode: {FILE_SELECTION_MODE}")
print(f"Keep last window real duration: {KEEP_LAST_WINDOW_REAL_DURATION}")
print(f"Write NaN sidecar: {WRITE_NAN_SIDECAR}")

if FILE_SELECTION_MODE == "range":
    print(f"Requested file range (1-based): n={FILE_RANGE_N}, p={FILE_RANGE_P}")

## 1. Discover & sort EDF files

In [ ]:
def parse_edf_files(prefix: str) -> list[tuple[int, Path]]:
    pattern = re.compile(rf"^{prefix}(\d+)\.edf$", re.IGNORECASE)
    files = []
    for entry in DATA_DIR.iterdir():
        m = pattern.match(entry.name)
        if m:
            unix_ts = int(m.group(1))
            files.append((unix_ts, entry))
    files.sort(key=lambda x: x[0])
    return files


def select_file_range(
    file_list: list[tuple[int, Path]],
    mode: str,
    n: int,
    p: int,
    label: str,
) -> list[tuple[int, Path]]:
    """Select either all files or only n..p (1-based, inclusive)."""
    total = len(file_list)
    if mode == "all":
        print(f"{label}: using all {total} file(s)")
        return file_list

    if mode != "range":
        raise ValueError(f"Invalid FILE_SELECTION_MODE={mode!r}; use 'all' or 'range'.")

    if total == 0:
        print(f"{label}: no files found")
        return file_list

    if n < 1 or p < 1:
        raise ValueError("FILE_RANGE_N and FILE_RANGE_P must be >= 1 (1-based indexing).")
    if n > p:
        raise ValueError(f"Invalid range: n ({n}) must be <= p ({p}).")

    start_idx = n - 1
    end_idx = p  # inclusive p in 1-based -> exclusive in slicing

    if start_idx >= total:
        raise ValueError(
            f"Range start n={n} is beyond available {label} files ({total})."
        )

    selected = file_list[start_idx:min(end_idx, total)]
    print(
        f"{label}: using files {n}..{min(p, total)} out of {total} "
        f"({len(selected)} selected)"
    )
    return selected

def detect_n_rats(file_list: list[tuple[int, Path]], ch_per_rat: int) -> int:
    "Open the first readable file to count channels and detect number of rats."
    if not file_list:
        return 0
    for _, fpath in file_list:
        try:
            reader = pyedflib.EdfReader(str(fpath))
        except OSError as exc:
            print(f" WARNING: cannot open {fpath.name}: {exc}")
            print(f"  → skipping this file for rat detection, trying the next one...")
            continue
        n_ch = reader.signals_in_file
        labels = reader.getSignalLabels()
        sample_rates = [reader.getSampleFrequency(i) for i in range(n_ch)]
        reader.close()
        if n_ch % ch_per_rat != 0:
            print(f" WARNING: {fpath.name} has {n_ch} channels, not a multiple of {ch_per_rat}!")
        n_rats = n_ch // ch_per_rat
        print(f" {fpath.name}: {n_ch} channels → {n_rats} rat(s)")
        print(f" Channel labels: {labels}")
        print(f" Sampling rates: {sample_rates}")
        return n_rats
    raise RuntimeError(":x: No readable EDF file found to detect rat count. Check your source folder.")

e_files_all = parse_edf_files("E")
a_files_all = parse_edf_files("A")

print(f"Found {len(e_files_all)} E-files and {len(a_files_all)} A-files")

e_files = select_file_range(e_files_all, FILE_SELECTION_MODE, FILE_RANGE_N, FILE_RANGE_P, "E-files")
a_files = select_file_range(a_files_all, FILE_SELECTION_MODE, FILE_RANGE_N, FILE_RANGE_P, "A-files")

if e_files:
    t0 = datetime.fromtimestamp(e_files[0][0], tz=LOCAL_TZ)
    t1 = datetime.fromtimestamp(e_files[-1][0], tz=LOCAL_TZ)
    print(f"E-files selected span: {t0} → {t1}")

if a_files:
    t0 = datetime.fromtimestamp(a_files[0][0], tz=LOCAL_TZ)
    t1 = datetime.fromtimestamp(a_files[-1][0], tz=LOCAL_TZ)
    print(f"A-files selected span: {t0} → {t1}")

print("\nDetecting number of rats...")
n_rats_e = detect_n_rats(e_files, N_CH_PER_RAT_E)
n_rats_a = detect_n_rats(a_files, N_CH_PER_RAT_A)
n_rats = max(n_rats_e, n_rats_a)

if n_rats_e != n_rats_a and n_rats_e > 0 and n_rats_a > 0:
    print(
        f"\n WARNING: E-files report {n_rats_e} rat(s) but A-files report {n_rats_a} rat(s).\n"
        f"   E-files and A-files are always recorded together per rat by the acquisition\n"
        f"   system, so this mismatch is unexpected and may indicate a missing or corrupt\n"
        f"   file, or a data acquisition issue. Check your source folder before continuing.\n"
        f"   Proceeding with {n_rats} rat(s) (max of both counts)."
    )

print(f"\n→ Working with {n_rats} rat(s)")

## 2. Group files into 24h windows

Each window is exactly 24 hours, anchored to the first file's timestamp. Window 0: `[T0, T0+24h)`, Window 1: `[T0+24h, T0+48h)`, etc.

In [ ]:
def group_by_24h_windows(
    file_list: list[tuple[int, Path]]
) -> list[tuple[int, int, list[tuple[int, Path]]]]:
    """
    Group files into fixed 24h windows starting from the first file's timestamp.

    Returns a list of (window_start_ts, window_end_ts, files_in_window).

    Important: include any file whose [start, start+EXPECTED_DURATION) interval
    overlaps the window. This keeps carry-over data at boundaries and avoids
    artificial gaps at the beginning of a window.
    """
    if not file_list:
        return []

    origin_ts = file_list[0][0]
    data_end_ts = file_list[-1][0] + EXPECTED_DURATION

    total_span = data_end_ts - origin_ts
    n_windows = int(np.ceil(total_span / WINDOW_DURATION))

    windows = []
    for w in range(n_windows):
        w_start = origin_ts + w * WINDOW_DURATION
        nominal_end = w_start + WINDOW_DURATION
        if KEEP_LAST_WINDOW_REAL_DURATION and w == n_windows - 1:
            w_end = data_end_ts
        else:
            w_end = nominal_end

        w_files = [
            (ts, fp) for ts, fp in file_list
            if (ts < w_end) and ((ts + EXPECTED_DURATION) > w_start)
        ]
        windows.append((w_start, w_end, w_files))

    return windows


e_windows = group_by_24h_windows(e_files)
a_windows = group_by_24h_windows(a_files)

print(f"E-files: {len(e_windows)} window(s)")
for i, (ws, we, wf) in enumerate(e_windows):
    t_start = datetime.fromtimestamp(ws, tz=LOCAL_TZ)
    t_end = datetime.fromtimestamp(we, tz=LOCAL_TZ)
    dur_h = (we - ws) / 3600.0
    print(f"  Window {i+1}: {t_start.strftime('%Y-%m-%d %H:%M')} → {t_end.strftime('%Y-%m-%d %H:%M')}  ({dur_h:.2f}h, {len(wf)} files)")

print(f"\nA-files: {len(a_windows)} window(s)")
for i, (ws, we, wf) in enumerate(a_windows):
    t_start = datetime.fromtimestamp(ws, tz=LOCAL_TZ)
    t_end = datetime.fromtimestamp(we, tz=LOCAL_TZ)
    dur_h = (we - ws) / 3600.0
    print(f"  Window {i+1}: {t_start.strftime('%Y-%m-%d %H:%M')} → {t_end.strftime('%Y-%m-%d %H:%M')}  ({dur_h:.2f}h, {len(wf)} files)")

## 3. Helper functions (pyedflib-based, native sampling rates)

In [ ]:
import json as _json   # used only by read_written_output_info / summary

def read_edf_channels(fpath: Path,
                      ch_indices: list[int] | None = None) -> list[dict] | None:
    """
    Read an EDF file with pyedflib, returning each channel at its NATIVE
    sampling rate.

    Parameters
    ----------
    fpath : Path
        Path to the EDF file.
    ch_indices : list[int] | None
        If provided, read only the channels at these positions (0-based).
        If None, all channels are read.

    Returns a list of dicts, one per selected channel:
        {'label': str, 'fs': float, 'data': np.ndarray}

    If the file is unreadable/non-compliant, returns None.
    """
    try:
        f = pyedflib.EdfReader(str(fpath))
    except OSError as exc:
        print(f"  Skipping unreadable EDF: {fpath.name} -- ({exc})")
        return None

    n_ch = f.signals_in_file
    indices = ch_indices if ch_indices is not None else list(range(n_ch))
    
    try:
        channels = []
        for i in indices:
            channels.append({
                'label': f.getLabel(i).strip(),
                'fs':    f.getSampleFrequency(i),
                'data':  f.readSignal(i),
            })
    finally:
        f.close()
    return channels

def build_window_channels(
    window_start: int,
    window_end: int,
    window_files: list[tuple[int, Path]],
    ch_indices: list[int] | None = None,
) -> list[dict]:
    """
    Build channel arrays for one window.

    Full windows are 24h. The last window can keep its real duration when
    KEEP_LAST_WINDOW_REAL_DURATION is enabled.

    Any gaps are filled with NaN; overlapping regions are overwritten by the
    later file.
    """
    if not window_files:
        return []

    first_channels = None
    first_file_idx = None
    for idx, (_, candidate_path) in enumerate(window_files):
        chs = read_edf_channels(candidate_path, ch_indices=ch_indices)
        if chs is not None:
            first_channels = chs
            first_file_idx = idx
            break

    if first_channels is None:
        print("  No readable EDF files in this window; skipping window")
        return []

    n_ch = len(first_channels)
    window_duration_s = window_end - window_start

    buffers = []
    for ch in first_channels:
        fs = ch['fs']
        total_samples = int(round(window_duration_s * fs))
        buffers.append({
            'label': ch['label'],
            'fs': fs,
            'data': np.full(total_samples, np.nan),
        })

    for file_idx, (unix_ts, fpath) in enumerate(window_files):
        if file_idx == first_file_idx:
            file_channels = first_channels
        else:
            file_channels = read_edf_channels(fpath, ch_indices=ch_indices)
            if file_channels is None:
                continue

        if len(file_channels) != n_ch:
            print(f"  Skipping {fpath.name}: channel count mismatch ({len(file_channels)} vs {n_ch})")
            continue

        offset_s = unix_ts - window_start

        for ch_idx in range(n_ch):
            fs = buffers[ch_idx]['fs']
            offset_samples = int(round(offset_s * fs))
            sig = file_channels[ch_idx]['data']
            n_sig = len(sig)

            buf_len = len(buffers[ch_idx]['data'])
            end = min(offset_samples + n_sig, buf_len)
            if offset_samples < 0:
                sig = sig[-offset_samples:]
                offset_samples = 0
            n_copy = end - offset_samples

            if n_copy > 0:
                buffers[ch_idx]['data'][offset_samples:end] = sig[:n_copy]

    for ch_idx, buf in enumerate(buffers):
        nan_count = np.isnan(buf['data']).sum()
        nan_pct = 100 * nan_count / len(buf['data']) if len(buf['data']) else 0.0
        dur = len(buf['data']) / buf['fs'] if buf['fs'] else 0.0
        if ch_idx == 0:
            print(f"  Window buffer: {dur:.0f}s ({dur/3600:.2f}h), {len(window_files)} files placed")
        if nan_pct > 0 and ch_idx == 0:
            print(f"  Missing data: {nan_pct:.1f}% NaN ({nan_count/buf['fs']:.0f}s of gaps)")

    return buffers

def split_channels_by_rat(channels: list[dict], ch_per_rat: int) -> list[list[dict]]:
    """
    Split a flat list of channel dicts into groups of ch_per_rat.
    Returns a list (per rat) of lists (per channel).
    """
    n_total = len(channels)
    n_rats = n_total // ch_per_rat
    return [
        channels[r * ch_per_rat : (r + 1) * ch_per_rat]
        for r in range(n_rats)
    ]

def local_datetime_str(unix_ts: int) -> tuple[str, str]:
    """Return (date_str, hour_str) in local timezone."""
    dt = datetime.fromtimestamp(unix_ts, tz=LOCAL_TZ)
    return dt.strftime("%Y-%m-%d"), dt.strftime("%Hh")


def build_output_path(rat_idx: int, signal_name: str, window_start: int) -> Path:
    """Return the expected EDF output path for one rat/signal/window."""
    date_str, hour_str = local_datetime_str(window_start)
    return OUTPUT_DIR / RAT_IDS[rat_idx - 1] / f"{signal_name}_{date_str}_{hour_str}.edf"


def read_written_output_info(edf_path: Path) -> dict | None:
    """Read timing metadata from an existing EDF or its JSON sidecar."""
    sidecar_path = edf_path.with_suffix(".json")
    if sidecar_path.exists():
        try:
            with open(sidecar_path) as fh:
                meta = _json.load(fh)
            fs = float(meta["fs"])
            n_samples = int(meta["n_samples"])
            start_ts = float(meta["starttime_unix"])
            return {
                "start_ts": start_ts,
                "duration_s": n_samples / fs,
                "metadata_source": "sidecar",
            }
        except Exception as exc:
            print(f"  Could not read sidecar {sidecar_path.name}: {exc}")

    if not edf_path.exists():
        return None

    try:
        reader = pyedflib.EdfReader(str(edf_path))
        try:
            start_dt = reader.getStartdatetime()
            if start_dt.tzinfo is None:
                start_dt = start_dt.replace(tzinfo=timezone.utc)
            else:
                start_dt = start_dt.astimezone(timezone.utc)
            duration_s = float(reader.getFileDuration())
        finally:
            reader.close()
    except OSError as exc:
        print(f"  Could not read existing EDF {edf_path.name}: {exc}")
        return None

    return {
        "start_ts": start_dt.timestamp(),
        "duration_s": duration_s,
        "metadata_source": "edf",
    }

def output_matches_window(
    edf_path: Path,
    expected_start: int,
    expected_duration: int | float,
) -> bool:
    """Return True when an existing output matches the expected window timing."""
    if not edf_path.exists():
        return False

    if WRITE_NAN_SIDECAR and not edf_path.with_suffix(".json").exists():
        print(
                f"  WARNING: '{edf_path.name} exists but its sidecar .json is missing. \n "
                f"  Likely an incomplete previous write. Will re-write this file."
        )
        return False

    info = read_written_output_info(edf_path)
    if info is None:
        print(
            f"  WARNING: {edf_path.name} exists but could not be read .\n"
            f"  May be corrupted. Will re-write this file."
            )
        return False

    start_tol_s = 1.0
    duration_tol_s = 1.0
    start_ok = abs(info["start_ts"] - float(expected_start)) <= start_tol_s
    duration_ok = abs(info["duration_s"] - float(expected_duration)) <= duration_tol_s
    return start_ok and duration_ok


def build_output_specs(kind: str, n_rats: int) -> list[dict]:
    """Describe the expected outputs for E-files or A-files."""
    specs = []
    if kind == "E":
        for rat_idx in range(1, n_rats + 1):
            for signal_name in CHANNEL_NAMES_E:
                specs.append({
                    "tag": f"rat{rat_idx}/{signal_name}",
                    "rat_idx": rat_idx,
                    "signal_name": signal_name,
                })
        return specs

    if kind == "A":
        for rat_idx in range(1, n_rats + 1):
            specs.append({
                "tag": f"rat{rat_idx}/Activity",
                "rat_idx": rat_idx,
                "signal_name": "Activity",
            })
        return specs

    raise ValueError(f"Unsupported output kind: {kind!r}")


def format_window_numbers(window_numbers: list[int]) -> str:
    """Format 1-based window numbers as compact ranges."""
    if not window_numbers:
        return "none"

    ordered = sorted(set(window_numbers))
    ranges = []
    start = ordered[0]
    prev = ordered[0]

    for number in ordered[1:]:
        if number == prev + 1:
            prev = number
            continue

        ranges.append(f"{start}" if start == prev else f"{start}-{prev}")
        start = prev = number

    ranges.append(f"{start}" if start == prev else f"{start}-{prev}")
    return ", ".join(ranges)


def summarize_window_plan(
    windows: list[tuple[int, int, list[tuple[int, Path]]]],
    pending_windows: list[tuple[int, int, int, list[tuple[int, Path]]]],
    label: str,
) -> None:
    """Print a concise summary of which windows will be skipped or processed."""
    non_empty_numbers = [idx + 1 for idx, (_, _, w_files) in enumerate(windows) if w_files]
    pending_numbers = [idx + 1 for idx, _, _, _ in pending_windows]
    skipped_numbers = [number for number in non_empty_numbers if number not in set(pending_numbers)]

    print(f"{label} plan:")
    print(f"  skip existing windows: {format_window_numbers(skipped_numbers)}")
    print(f"  process windows: {format_window_numbers(pending_numbers)}")


def plan_incremental_windows(
    windows: list[tuple[int, int, list[tuple[int, Path]]]],
    output_specs: list[dict],
    label: str,
) -> tuple[list[tuple[int, int, int, list[tuple[int, Path]]]], dict[str, int | None]]:
    """
    Compare source windows to outputs already on disk and return only windows
    that still need to be written or refreshed.
    """
    last_written_window = {spec["tag"]: None for spec in output_specs}
    pending_windows = []
    skipped_complete = 0
    skipped_empty = 0

    for w_idx, (w_start, w_end, w_files) in enumerate(windows):
        if not w_files:
            skipped_empty += 1
            continue

        expected_duration = w_end - w_start
        window_complete = True

        for spec in output_specs:
            out_path = build_output_path(spec["rat_idx"], spec["signal_name"], w_start)
            if output_matches_window(out_path, w_start, expected_duration):
                last_written_window[spec["tag"]] = w_start
            else:
                window_complete = False

        if window_complete:
            skipped_complete += 1
        else:
            pending_windows.append((w_idx, w_start, w_end, w_files))

    print(f"{label}: {skipped_complete} complete window(s) already on disk")
    if skipped_empty:
        print(f"{label}: {skipped_empty} empty source window(s) ignored")
    print(f"{label}: {len(pending_windows)} window(s) still need processing")

    for spec in output_specs:
        last_start = last_written_window[spec["tag"]]
        if last_start is None:
            print(f"  {spec['tag']}: no matching written window found")
            continue

        dt_last = datetime.fromtimestamp(last_start, tz=LOCAL_TZ)
        print(f"  {spec['tag']}: last matching window starts {dt_last.strftime('%Y-%m-%d %H:%M')}")

    return pending_windows, last_written_window

## 4. Create rat subdirectories

In [ ]:
for r in range(1, n_rats + 1):
    (OUTPUT_DIR / RAT_IDS[r - 1]).mkdir(exist_ok=True)
    print(f"Created: {OUTPUT_DIR / RAT_IDS[r - 1]}")

## 5. Incremental plan

Summarize which windows already exist on disk and which ones will be written on this run.

In [ ]:
print("=" * 55)
print(" INCREMENTAL PLAN - E-files")
print("=" * 55)
e_output_specs = build_output_specs("E", n_rats)
e_pending_windows, e_last_written_windows = plan_incremental_windows(
    e_windows,
    e_output_specs,
    "E-files",
)
summarize_window_plan(e_windows, e_pending_windows, "E-files")

print()
print("=" * 55)
print(" INCREMENTAL PLAN - A-files")
print("=" * 55)
a_output_specs = build_output_specs("A", n_rats)
a_pending_windows, a_last_written_windows = plan_incremental_windows(
    a_windows,
    a_output_specs,
    "A-files",
)
summarize_window_plan(a_windows, a_pending_windows, "A-files")

## 6. Process E-files (parallelized writes)

Each pending 24h window is read and assembled into a fixed-length buffer,
then per-rat / per-channel EDF writes are dispatched to the process pool.

Each channel is written at its native sampling rate.

In [ ]:
t_start = time.perf_counter()
n_written = 0
n_errors = 0

print()
print(f"Processing E-files → individual channel files ({N_WORKERS} workers)")
print("=" * 65)

if not e_pending_windows:
    print("No new or changed E-file windows to process.")
else:
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        for window_idx, w_start, w_end, w_files in e_pending_windows:
            date_str, hour_str = local_datetime_str(w_start)
            dt_start = datetime.fromtimestamp(w_start, tz=LOCAL_TZ)
            dt_end = datetime.fromtimestamp(w_end, tz=LOCAL_TZ)
            dur_h = (w_end - w_start) / 3600.0
            print(f"\nWindow {window_idx + 1}: {dt_start.strftime('%Y-%m-%d %H:%M')} → "
                  f"{dt_end.strftime('%Y-%m-%d %H:%M')} ({dur_h:.2f}h, {len(w_files)} files)")

            print( f" Reading {len(w_files)} source files intro memory ...", flush=True)
            channels_all = build_window_channels(w_start, w_end, w_files)
            print("=" * 55)
            print(f" Done reading. Dispatching writes...", flush=True)

            rat_channel_groups = split_channels_by_rat(channels_all, N_CH_PER_RAT_E)

            if len(rat_channel_groups) != n_rats:
                print( f" WARMING: expected {n_rats} rat group(s) but got"
                       f"{len(rat_channel_groups)}.Some rats may be missing output files")
            futures = {}
            for r, rat_channels in enumerate(rat_channel_groups, start=1):
                for ch_idx, ch_dict in enumerate(rat_channels):
                    ch_name = CHANNEL_NAMES_E[ch_idx]
                    ch_data = ch_dict['data']
                    ch_fs = ch_dict['fs']
                    ch_data, cal = calibration.calibrate(ch_name, ch_data, RECEIVER_ID, r, TEMPERATURE_TABLE)
                    fname = f"{ch_name}_{date_str}_{hour_str}.edf"
                    out_path = OUTPUT_DIR / RAT_IDS[r - 1] / fname
                    sidecar_path = out_path.with_suffix(".json") if WRITE_NAN_SIDECAR else None

                    fut = pool.submit(
                        write_single_channel_edf,
                        str(out_path),
                        ch_data,
                        ch_fs,
                        ch_name,
                        float(w_start),
                        str(sidecar_path) if sidecar_path is not None else None,
                        **cal,
                    )
                    futures[fut] = f"rat{r}/{fname} ({int(ch_fs)} Hz)"

            for fut in as_completed(futures):
                tag = futures[fut]
                try:
                    msg = fut.result()
                    n_written += 1
                    print(f"  {tag}: {msg}")
                except Exception as exc:
                    n_errors += 1
                    print(f" ERROR {tag}: {exc}")

            del channels_all, rat_channel_groups

t_end = time.perf_counter()
print(f"\n E-files done: {n_written} written, {n_errors} errors, {t_end - t_start:.1f}s total")

## 7. Process A-files (parallelized writes)

In [ ]:
t_start = time.perf_counter()
n_written = 0
n_errors = 0

print()
print(f"Processing A-files → Activity ({N_WORKERS} workers)")
print("=" * 65)

if not a_pending_windows:
    print("No new or changed A-file windows to process.")
else:
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        for window_idx, w_start, w_end, w_files in a_pending_windows:
            date_str, hour_str = local_datetime_str(w_start)
            dt_start = datetime.fromtimestamp(w_start, tz=LOCAL_TZ)
            dt_end = datetime.fromtimestamp(w_end, tz=LOCAL_TZ)
            dur_h = (w_end - w_start) / 3600.0
            print(f"\nWindow {window_idx + 1}: {dt_start.strftime('%Y-%m-%d %H:%M')} → "
                  f"{dt_end.strftime('%Y-%m-%d %H:%M')} ({dur_h:.2f}h, {len(w_files)} files)")

            print( f" Reading {len(w_files)} source files intro memory ...", flush=True)
            channels_all = build_window_channels(w_start, w_end, w_files)
            print("=" * 55)
            print(f" Done reading. Dispatching writes...", flush=True)
            rat_channel_groups = split_channels_by_rat(channels_all, N_CH_PER_RAT_A)

            futures = {}
            for r, rat_channels in enumerate(rat_channel_groups, start=1):
                ch_data = rat_channels[0]['data']
                ch_fs = rat_channels[0]['fs']
                ch_data, cal = calibration.calibrate("Activity", ch_data, RECEIVER_ID, r, TEMPERATURE_TABLE)
                fname = f"Activity_{date_str}_{hour_str}.edf"
                out_path = OUTPUT_DIR / RAT_IDS[r - 1] / fname
                sidecar_path = out_path.with_suffix(".json") if WRITE_NAN_SIDECAR else None

                fut = pool.submit(
                    write_single_channel_edf,
                    str(out_path),
                    ch_data,
                    ch_fs,
                    "Activity",
                    float(w_start),
                    str(sidecar_path) if sidecar_path is not None else None,
                    **cal,
                )
                futures[fut] = f"rat{r}/{fname} ({int(ch_fs)} Hz)"

            for fut in as_completed(futures):
                tag = futures[fut]
                try:
                    msg = fut.result()
                    n_written += 1
                    print(f"  {tag}: {msg}")
                except Exception as exc:
                    n_errors += 1
                    print(f"  {tag}: {exc}")

            del channels_all, rat_channel_groups

t_end = time.perf_counter()
print(f"\n A-files done: {n_written} written, {n_errors} errors, {t_end - t_start:.1f}s total")

## 8. Summary

In [ ]:
print(f"Output directory: {OUTPUT_DIR}")
print()

for r in range(1, n_rats + 1):
    rat_dir = OUTPUT_DIR / RAT_IDS[r - 1]
    edf_files = sorted(rat_dir.glob("*.edf"))
    total_mb = sum(edf_path.stat().st_size for edf_path in edf_files) / 1e6
    print(f"{RAT_IDS[r - 1]}/  ({len(edf_files)} EDF files, {total_mb:.1f} MB total)")

    for edf_path in edf_files:
        size_mb = edf_path.stat().st_size / 1e6
        sidecar = edf_path.with_suffix(".json")

        if sidecar.exists():
            with open(sidecar) as fh:
                meta = _json.load(fh)
            fs = float(meta["fs"])
            n_samples = int(meta["n_samples"])
            gaps = meta.get("gaps", [])
            duration_h = n_samples / fs / 3600.0
            missing_s = sum(g[1] for g in gaps)
            missing_pct = 100.0 * missing_s / (n_samples / fs) if n_samples else 0.0
            sidecar_name = sidecar.name
            print(
                f"  {edf_path.name:45s}  {size_mb:6.1f} MB | "
                f"duration={duration_h:6.2f} h | missing={missing_pct:5.1f}% | sidecar={sidecar_name}"
            )
        else:
            print(f"  {edf_path.name:45s}  {size_mb:6.1f} MB | sidecar=missing")

    print()

In [ ]:
# Run all cells until here to dont run the verification section
RUN_REST = False 

if not RUN_REST:
    raise SystemExit("Notebook stopped here")

## VERIFICATION SECTION
Inspect actual source file durations and spacing

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.rcParams["figure.dpi"] = 110

# ── CONFIG: how many files to inspect (from the start of e_files) ─────────────
VERIF_N_FILES = min(6, len(e_files))   # <── change this to inspect more or fewer files

files_to_inspect = e_files[:VERIF_N_FILES]
print(f"Inspecting first {len(files_to_inspect)} E-file(s):")
print(f"{'#':<4} {'Filename':<24} {'Unix TS':>12}  {'Local time':<22}  "
      f"{'Header dur (s)':>14}  {'Gap to next (s)':>16}  {'Note'}")
print("-" * 105)

prev_ts = None
for i, (ts, fp) in enumerate(files_to_inspect):
    try:
        reader = pyedflib.EdfReader(str(fp))
        try:
            actual_dur = float(reader.getFileDuration())
        finally:
            reader.close()
    except Exception as exc:
        actual_dur = float("nan")
        note = f"WARNING -> UNREADABLE: {exc}"

    gap_to_next = ""
    note = ""
    if prev_ts is not None:
        gap = ts - prev_ts
        gap_to_next = f"{gap:>16.0f}"
        if abs(gap - EXPECTED_DURATION) > 30:
            note = f"WARNING ->  gap≠{EXPECTED_DURATION}s (diff={gap - EXPECTED_DURATION:+.0f}s)"
    else:
        gap_to_next = f"{'(first)':>16}"

    dur_note = ""
    if not (EXPECTED_DURATION - 30 <= actual_dur <= EXPECTED_DURATION + 30):
        dur_note = f"  ← WARNING ->  expected ~{EXPECTED_DURATION}s"

    dt_local = datetime.fromtimestamp(ts, tz=LOCAL_TZ).strftime("%Y-%m-%d %H:%M:%S")
    print(f"{i+1:<4} {fp.name:<24} {ts:>12}  {dt_local:<22}  "
          f"{actual_dur:>14.1f}{dur_note}  {gap_to_next}  {note}")
    prev_ts = ts

print()
print("Legend:")
print(f"  'Header dur (s)' — actual duration recorded in the EDF header.")
print(f"  'Gap to next (s)' — seconds between this file's start and the next file's start.")
print(f"  Expected both to be ≈ {EXPECTED_DURATION} s. Warnings appear if deviation > 30 s.")


Timeline plot of files in window

In [ ]:
import ipywidgets as widgets
from IPython.display import display
matplotlib.use("module://matplotlib_inline.backend_inline")

# ── 1. First overlap ────────────────────────────────────────────────────────
def plot_file_overlap(pair_idx: int, context_s: float = 10.0):
    if pair_idx >= len(e_files) - 1:
        print(f"pair_idx={pair_idx} invalid, only {len(e_files)-1} pairs available.")
        return

    ts1, fp1 = e_files[pair_idx]
    ts2, fp2 = e_files[pair_idx + 1]
    overlap_s = (ts1 + EXPECTED_DURATION) - ts2

    if overlap_s <= 0:
        print(f"No overlap — gap of {-overlap_s:.0f}s between {fp1.name} and {fp2.name}.")
        return

    chs1 = read_edf_channels(fp1)
    chs2 = read_edf_channels(fp2)
    if chs1 is None or chs2 is None:
        print("Unreadable file.")
        return

    fs            = chs1[0]['fs']
    overlap_samp  = int(overlap_s * fs)
    context_samp  = int(context_s * fs)

    tail1 = chs1[0]['data'][-(context_samp + overlap_samp):]
    head2 = chs2[0]['data'][:overlap_samp + context_samp]

    t1_start     = EXPECTED_DURATION - context_s - overlap_s
    t1_abs       = np.arange(len(tail1)) / fs + t1_start
    t2_start     = ts2 - ts1
    t2_abs       = np.arange(len(head2)) / fs + t2_start
    overlap_start = float(ts2 - ts1)
    overlap_end   = float(EXPECTED_DURATION)

    diff   = tail1[-overlap_samp:] - head2[:overlap_samp]
    t_diff = np.arange(overlap_samp) / fs + overlap_start

    plt.close("all")
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    fig.suptitle(
        f"Pair {pair_idx+1}→{pair_idx+2}: {fp1.name} | {fp2.name}  "
        f"— channel '{chs1[0]['label']}' @ {fs}Hz — overlap={overlap_s:.0f}s",
        fontsize=10
    )

    ax = axes[0]
    ax.plot(t1_abs, tail1, color="steelblue", lw=0.6, label=f"File {pair_idx+1}: {fp1.name}")
    ax.plot(t2_abs, head2, color="darkorange", lw=0.6, alpha=0.8, label=f"File {pair_idx+2}: {fp2.name}")
    ax.axvspan(overlap_start, overlap_end, color="red", alpha=0.12, label=f"Overlap ({overlap_s:.0f}s)")
    ax.axvline(overlap_start, color="red", lw=1, linestyle="--")
    ax.axvline(overlap_end,   color="red", lw=1, linestyle="--")
    ax.set_xlim(t1_start - 1, t2_start + overlap_s + context_s + 1)
    ax.set_ylabel("Signal value", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    ax = axes[1]
    ax.plot(t_diff, diff, color="crimson", lw=0.6)
    ax.axhline(0, color="black", lw=0.8, linestyle="--")
    ax.fill_between(t_diff, diff, 0, alpha=0.25, color="crimson")
    ax.set_xlim(t1_start - 1, t2_start + overlap_s + context_s + 1)
    ax.set_ylabel("File1 − File2", fontsize=9)
    ax.set_xlabel("Time from start of File 1 (seconds)", fontsize=9)
    ax.set_title(
        f"Difference — max={diff.max():.1f}, min={diff.min():.1f}, mean={diff.mean():.3f}",
        fontsize=10
    )
    ax.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.show()

# ── 2. Widget setup ─────────────────────────────────────────────────────────
pair_options = [
    (f"Pair {i+1}→{i+2}:  {e_files[i][1].name}  →  {e_files[i+1][1].name}", i)
    for i in range(len(e_files) - 1)
]

dropdown = widgets.Dropdown(
    options=pair_options,
    value=0,
    description="Pair:",
    layout=widgets.Layout(width="500px")
)

out = widgets.Output()

def on_change(change):
    if change["name"] == "value":
        out.clear_output(wait=True)
        with out:
            plot_file_overlap(pair_idx=change["new"])

dropdown.observe(on_change)
display(dropdown, out)

with out:
    plot_file_overlap(pair_idx=0)

In [ ]:
ts1, fp1 = e_files[2]
ts2, fp2 = e_files[3]

r1 = pyedflib.EdfReader(str(fp1))
r2 = pyedflib.EdfReader(str(fp2))
print(f"File 1 real duration: {r1.getFileDuration():.6f}s")
print(f"File 2 real duration: {r2.getFileDuration():.6f}s")
print(f"Expected duration: {EXPECTED_DURATION}s")
r1.close()
r2.close()

In [ ]:
for i in range(len(e_files) - 1):
    ts1, fp1 = e_files[i]
    ts2, fp2 = e_files[i + 1]
    overlap_s = (ts1 + EXPECTED_DURATION) - ts2
    if overlap_s <= 0:
        continue
    chs1 = read_edf_channels(fp1)
    chs2 = read_edf_channels(fp2)
    fs = chs1[0]['fs']
    overlap_samp = int(overlap_s * fs)
    diff = chs1[0]['data'][-overlap_samp:] - chs2[0]['data'][:overlap_samp]
    print(f"Pair {i+1}→{i+2}: overlap={overlap_s}s  "
          f"max_diff={np.abs(diff).max():.1f}  mean_diff={diff.mean():.4f}")

Manual buffer assembly vs pipeline

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
VERIF_WINDOW_IDX = 10  # 0-based window index
VERIF_RAT        = 1   # 1-based rat number
VERIF_CH_IDX     = 4   # 0-based channel index within the rat (0=EMG, 1=ECG, 2=EGG, 3=EEG, 4=Temp)
# ─────────────────────────────────────────────────────────────────────────────

ch_name_v, ch_fs_v = CHANNELS_PER_RAT_E[VERIF_CH_IDX]
ch_global_v = (VERIF_RAT - 1) * N_CH_PER_RAT_E + VERIF_CH_IDX

w_start_v, w_end_v, w_files_v = e_windows[VERIF_WINDOW_IDX]
w_dur_v = w_end_v - w_start_v
total_samples_v = int(round(w_dur_v * ch_fs_v))

print(f"Window {VERIF_WINDOW_IDX + 1}  |  rat {VERIF_RAT}  |  channel '{ch_name_v}' "
      f"(index {VERIF_CH_IDX}, fs={ch_fs_v} Hz, global idx={ch_global_v})")
print(f"Buffer size: {total_samples_v} samples  ({w_dur_v}s = {w_dur_v/3600:.2f}h)")
print(f"Files in window: {len(w_files_v)}")
print()

# ── Manual assembly ───────────────────────────────────────────────────────────
manual_buf = np.full(total_samples_v, np.nan)

print(f"  {'File':<28}  {'Offset (s)':>10}  {'File dur (s)':>12}  {'Samples read':>13}  {'Samples copied':>14}")
for ts, fp in w_files_v:
    chs = read_edf_channels(fp, ch_indices=[ch_global_v])
    if chs is None:
        print(f"  {fp.name:<28}  UNREADABLE — skipped")
        continue
    sig      = chs[0]["data"]
    offset_s = ts - w_start_v
    off_samp = int(round(offset_s * ch_fs_v))
    n_sig    = len(sig)
    buf_end  = min(off_samp + n_sig, total_samples_v)
    sig_start = 0
    if off_samp < 0:
        sig_start = -off_samp
        off_samp  = 0
    n_copy = buf_end - off_samp
    if n_copy > 0:
        manual_buf[off_samp:buf_end] = sig[sig_start : sig_start + n_copy]

    try:
        reader = pyedflib.EdfReader(str(fp))
        file_dur_hdr = float(reader.getFileDuration())
        reader.close()
    except Exception:
        file_dur_hdr = float("nan")

    print(f"  {fp.name:<28}  {offset_s:>10.1f}  {file_dur_hdr:>12.1f}  {n_sig:>13}  {n_copy:>14}")

nan_man = np.isnan(manual_buf).sum()
print(f"\nManual buffer: {total_samples_v} samples, "
      f"{nan_man} NaN ({100*nan_man/total_samples_v:.2f}%  ≈ {nan_man/ch_fs_v:.1f}s missing)")

# ── Pipeline assembly ─────────────────────────────────────────────────────────
print("\nRunning build_window_channels (pipeline)...")
pipeline_bufs = build_window_channels(w_start_v, w_end_v, w_files_v)
pip_ch        = pipeline_bufs[ch_global_v]
pipeline_buf  = pip_ch["data"]
nan_pip       = np.isnan(pipeline_buf).sum()
print(f"Pipeline buffer: {len(pipeline_buf)} samples, "
      f"{nan_pip} NaN ({100*nan_pip/len(pipeline_buf):.2f}%)")

# ── Sample-by-sample comparison ───────────────────────────────────────────────
print()
n_cmp   = min(len(manual_buf), len(pipeline_buf))
m_trim  = manual_buf[:n_cmp]
p_trim  = pipeline_buf[:n_cmp]

len_ok  = len(manual_buf) == len(pipeline_buf)
nan_ok  = np.array_equal(np.isnan(m_trim), np.isnan(p_trim))
both_v  = ~np.isnan(m_trim) & ~np.isnan(p_trim)
val_ok  = bool(np.all(m_trim[both_v] == p_trim[both_v])) if both_v.sum() > 0 else True

print(f"Length match : {'YES' if len_ok else 'NO  (manual=' + str(len(manual_buf)) + ', pipeline=' + str(len(pipeline_buf)) + ')'}")
print(f"NaN placement: {'IDENTICAL' if nan_ok else 'DIFFER'}")
print(f"Signal values: {'IDENTICAL' if val_ok else ' DIFFER'}")
if len_ok and nan_ok and val_ok:
    print("\n VERIFIED: manual and pipeline buffers are bit-for-bit identical.")
else:
    print("\n MISMATCH — see the plot below for details.")

In [ ]:
# Plot: manual vs pipeline signal + difference
t_h = np.arange(n_cmp) / ch_fs_v / 3600.0

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig.suptitle(
    f"Verification: '{ch_name_v}' — rat {VERIF_RAT} — Window {VERIF_WINDOW_IDX + 1}\n"
    f"(window start: {datetime.fromtimestamp(w_start_v, tz=LOCAL_TZ).strftime('%Y-%m-%d %H:%M %Z')})",
    fontsize=10
)
# -- panel 1: manual --
ax = axes[0]
valid_m = ~np.isnan(m_trim)
ax.plot(t_h[valid_m], m_trim[valid_m], lw=0.4, color="steelblue")
for ts, _ in w_files_v:
    ax.axvline((ts - w_start_v) / 3600, color="orange", lw=0.8, linestyle="--", alpha=0.7)
ax.set_ylabel("Manual\nassembly", fontsize=8)
ax.grid(True, alpha=0.25)
# shade NaN gaps
_in, _gs = False, 0.0
for i in range(n_cmp):
    if np.isnan(m_trim[i]) and not _in:
        _gs = t_h[i]; _in = True
    elif not np.isnan(m_trim[i]) and _in:
        ax.axvspan(_gs, t_h[i], alpha=0.25, color="red")
        _in = False
if _in:
    ax.axvspan(_gs, t_h[-1], alpha=0.25, color="red")

# -- panel 2: pipeline --
ax = axes[1]
valid_p = ~np.isnan(p_trim)
ax.plot(t_h[valid_p], p_trim[valid_p], lw=0.4, color="darkorange")
for ts, _ in w_files_v:
    ax.axvline((ts - w_start_v) / 3600, color="steelblue", lw=0.8, linestyle="--", alpha=0.7)
ax.set_ylabel("Pipeline\nbuild_window_channels", fontsize=8)
ax.grid(True, alpha=0.25)

# -- panel 3: difference --
ax = axes[2]
diff = np.where(~np.isnan(m_trim) & ~np.isnan(p_trim), m_trim - p_trim, np.nan)
has_diff = np.any(diff[~np.isnan(diff)] != 0) if not np.all(np.isnan(diff)) else False
ax.plot(t_h, diff, lw=0.4, color="crimson" if has_diff else "green")
if not has_diff:
    ax.text(0.5, 0.5, "Zero difference — buffers are identical",
            transform=ax.transAxes, ha="center", va="center",
            fontsize=11, color="green", fontweight="bold")
ax.set_ylabel("Difference\n(manual − pipeline)", fontsize=8)
ax.set_xlabel("Time from window start (hours)", fontsize=9)
ax.grid(True, alpha=0.25)

orange_line = mpatches.Patch(color="orange", alpha=0.7, label="File start (manual panel)")
blue_line   = mpatches.Patch(color="steelblue", alpha=0.7, label="File start (pipeline panel)")
red_patch   = mpatches.Patch(color="red", alpha=0.25, label="NaN gap (missing data)")
axes[0].legend(handles=[orange_line, red_patch], fontsize=7, loc="upper right")

plt.tight_layout()
plt.show()